# CareLoop Clinics — No-Show Guard
## Part 3 · Predict no-shows and choose the model honestly (Tasks 3.1–3.5)

**Input:** `appointments_clean.csv` (Part 1), `clinics_enriched.csv` (Section 0 cross-check)
**Output:** `test_risk_scores.csv`

**Rule for this whole notebook:** the test set is only for reporting. Model, settings and
threshold are all chosen with training data and cross-validation.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)

df = pd.read_csv("appointments_clean.csv")      # row order is kept as saved (sorted by appointment_id)
print(df.shape)

(990, 16)


## 0 · Python cross-check of Task 2.2 (clinic counts vs SQL)
The SQL in `02_queries.sql` returned these counts; groupby must reproduce them exactly.

In [2]:
clinics = pd.read_csv("clinics_enriched.csv")
by_clinic = (df.merge(clinics[["clinic_id", "clinic_name", "city"]], on="clinic_id")
               .groupby(["clinic_name", "city"])
               .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum")))
by_clinic["no_show_pct"] = (100 * by_clinic["no_shows"] / by_clinic["appointments"]).round(1)
by_clinic = by_clinic.sort_values("no_show_pct", ascending=False)
print(by_clinic)

sql_counts = {  # copied from the SQL output of Task 2.2 (no HAVING)
    "Gomti Nagar Care Point": (90, 30), "Hadapsar Care Centre": (143, 39),
    "Malviya Nagar Clinic": (142, 38), "Kothrud Family Clinic": (160, 36),
    "Vijay Nagar Clinic": (101, 21), "Edappally Medical Centre": (142, 23),
    "Sitabuldi Health Point": (82, 13), "Vaishali Nagar Clinic": (130, 18)}
py_counts = {name: (int(r.appointments), int(r.no_shows))
             for (name, _), r in by_clinic.iterrows()}
print("\npandas counts match SQL counts:", py_counts == sql_counts)

                                  appointments  no_shows  no_show_pct
clinic_name              city                                        
Gomti Nagar Care Point   Lucknow            90        30         33.3
Hadapsar Care Centre     Pune              143        39         27.3
Malviya Nagar Clinic     Jaipur            142        38         26.8
Kothrud Family Clinic    Pune              160        36         22.5
Vijay Nagar Clinic       Indore            101        21         20.8
Edappally Medical Centre Kochi             142        23         16.2
Sitabuldi Health Point   Nagpur             82        13         15.9
Vaishali Nagar Clinic    Jaipur            130        18         13.8

pandas counts match SQL counts: True


## Task 3.1 — Features, split and baseline

In [3]:
FEATURES = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
            "reminder_sent", "booking_channel", "visit_type"]
X = df[FEATURES].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], dtype=int)
y = df["no_show"]

print("X columns:", X.shape[1])
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTrain: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"No-shows in test set: {y_test.sum()} | overall no-show share: {y.mean() * 100:.1f}%")

baseline = (y_test == 0).mean()
print(f"Baseline accuracy (always predict 'Attended'): {baseline:.4f} "
      f"({(y_test == 0).sum()} of {len(y_test)})")

X columns: 11
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Train: 792 rows | Test: 198 rows
No-shows in test set: 49 | overall no-show share: 22.0%
Baseline accuracy (always predict 'Attended'): 0.7525 (149 of 198)


## Task 3.2 — Logistic regression

In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)        # fitted on training rows only
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

def report(name, y_true, pred, prob):
    m = {"model": name,
         "accuracy": accuracy_score(y_true, pred),
         "precision": precision_score(y_true, pred, zero_division=0),
         "recall": recall_score(y_true, pred),
         "f1": f1_score(y_true, pred),
         "roc_auc": roc_auc_score(y_true, prob)}
    for k, v in m.items():
        print(f"{k:>10}: {v:.3f}" if isinstance(v, float) else f"{k:>10}: {v}")
    return m

lr_metrics = report("Logistic regression", y_test, lr_pred, lr_prob)
print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, lr_pred))

     model: Logistic regression
  accuracy: 0.788
 precision: 0.630
    recall: 0.347
        f1: 0.447
   roc_auc: 0.789

Confusion matrix [[TN, FP], [FN, TP]]:
[[139  10]
 [ 32  17]]


In [5]:
coefs = pd.DataFrame({"feature": X.columns, "coef_scaled": lr.coef_[0],
                      "train_sd": scaler.scale_})
print(coefs.round(3).to_string(index=False))

i = list(X.columns).index("reminder_sent")
coef_r = lr.coef_[0][i]
sd_r = scaler.scale_[i]
print(f"\nreminder_sent coefficient (scaled): {coef_r:.3f}")
print(f"np.exp(coef)          = {np.exp(coef_r):.2f}  -> odds multiplier per 1 SD (SD = {sd_r:.2f})")
print(f"np.exp(coef / scale)  = {np.exp(coef_r / sd_r):.2f}  -> odds multiplier for No (0) -> Yes (1)")
print(f"i.e. odds of a no-show about {(1 - np.exp(coef_r / sd_r)) * 100:.0f}% lower with a reminder")

                feature  coef_scaled  train_sd
                    age       -0.254    16.656
            distance_km        0.253     6.052
                fee_inr        0.453   184.254
              lead_days        0.614     8.614
         prior_no_shows        0.650     0.969
          reminder_sent       -0.581     0.495
    booking_channel_App       -0.047     0.486
  booking_channel_Phone        0.216     0.481
booking_channel_Walk-In       -0.186     0.435
visit_type_Consultation        0.016     0.493
   visit_type_Follow-up       -0.016     0.493

reminder_sent coefficient (scaled): -0.581
np.exp(coef)          = 0.56  -> odds multiplier per 1 SD (SD = 0.49)
np.exp(coef / scale)  = 0.31  -> odds multiplier for No (0) -> Yes (1)
i.e. odds of a no-show about 69% lower with a reminder


**Interpretation.** The two numbers differ because the model was fitted on standardised
features: `exp(coef)` is the odds change for a one-*standard-deviation* step in
`reminder_sent` (SD ≈ 0.49), whereas a real reminder is a full 0 → 1 step, about two SDs,
so the coefficient must be divided by the SD before exponentiating.

**Conclusion (association, not cause):** in this model, holding the other features
constant, appointments with a reminder have roughly 69% lower odds of a no-show
(odds × ~0.31). The data are observational, so this does not prove that sending a reminder
causes fewer no-shows; patients who receive reminders may differ in other ways.

`lead_days` and `prior_no_shows` carry the largest positive coefficients: longer lead times
and a history of missed visits are both associated with higher no-show odds.

## Task 3.3 — Decision tree and overfitting

In [6]:
tree_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_d3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

for name, t in [("Full-depth tree", tree_full), ("Depth-3 tree", tree_d3)]:
    tr = accuracy_score(y_train, t.predict(X_train))
    te = accuracy_score(y_test, t.predict(X_test))
    print(f"{name:16} depth={t.get_depth():2}  train acc={tr:.3f}  test acc={te:.3f}  gap={tr - te:.3f}")
print(f"Baseline accuracy: {baseline:.4f}")

Full-depth tree  depth=15  train acc=1.000  test acc=0.672  gap=0.328
Depth-3 tree     depth= 3  train acc=0.822  test acc=0.778  gap=0.044
Baseline accuracy: 0.7525


**What the gap shows:** the unlimited tree scores 100% on training data but falls below the
0.7525 always-"Attended" baseline on the test set, so it has memorised noise in the training
rows (overfitting); limiting depth to 3 shrinks the gap and generalises better.

## Task 3.4 — Random forest and comparison table

In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print(f"Training accuracy: {accuracy_score(y_train, rf.predict(X_train)):.3f}")
rf_metrics = report("Random forest", y_test, rf_pred, rf_prob)

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop three feature importances:")
print(imp.head(3).round(3).to_string())

Training accuracy: 1.000
     model: Random forest
  accuracy: 0.747
 precision: 0.484
    recall: 0.306
        f1: 0.375
   roc_auc: 0.706

Top three feature importances:
lead_days      0.220
age            0.197
distance_km    0.195


In [8]:
d3_metrics = report("Depth-3 tree", y_test, tree_d3.predict(X_test), tree_d3.predict_proba(X_test)[:, 1])
comparison = pd.DataFrame([lr_metrics, d3_metrics, rf_metrics])[["model", "accuracy", "recall", "roc_auc"]]
comparison.round(3)

     model: Depth-3 tree
  accuracy: 0.778
 precision: 0.647
    recall: 0.224
        f1: 0.333
   roc_auc: 0.700


,model,accuracy,recall,roc_auc
0,Logistic regression,0.788,0.347,0.789
1,Depth-3 tree,0.778,0.224,0.700
2,Random forest,0.747,0.306,0.706


**Reading the table (reporting only):** logistic regression looks best: it has the highest
test ROC-AUC, i.e. it ranks risky bookings above safe ones most reliably, and its recall is
competitive. The default random forest fits the training data perfectly but ranks test
bookings less well. These are test-set numbers, so they are **not** used to pick the model;
the formal choice is made in Task 3.5 from cross-validation on the training set.

## Task 3.5 — Tune, choose the model and threshold on training data, then test once
### (a) Grid search on the random forest

In [9]:
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print(f"best_score_ (cross-validated ROC-AUC): {grid.best_score_:.3f}")

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (cross-validated ROC-AUC): 0.767


### (b) Choose the model by 5-fold cross-validated ROC-AUC (training set only)
Logistic regression goes through a pipeline so the scaler is re-fitted inside each fold.

In [10]:
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

cv_scores = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Depth-3 tree": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                    X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100 trees)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                                 X_train, y_train, cv=5, scoring="roc_auc").mean(),
    f"Tuned forest {grid.best_params_}": grid.best_score_,
}
cv_table = pd.Series(cv_scores, name="cv_roc_auc").sort_values(ascending=False).round(3)
print(cv_table.to_string())
winner = cv_table.index[0]
print("\nWinner:", winner)

Logistic regression (pipeline)                        0.774
Tuned forest {'max_depth': 3, 'n_estimators': 200}    0.767
Random forest (100 trees)                             0.747
Depth-3 tree                                          0.678

Winner: Logistic regression (pipeline)


### (c) Choose the threshold from out-of-fold training probabilities

In [11]:
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, flag),
                 "recall": recall_score(y_train, flag),
                 "f1": f1_score(y_train, flag),
                 "flagged": int(flag.sum())})
thr_table = pd.DataFrame(rows)
print(thr_table.round(3).to_string(index=False))
THRESHOLD = float(thr_table.loc[thr_table["f1"].idxmax(), "threshold"])
print("\nChosen threshold (higher F1):", THRESHOLD)

 threshold  precision  recall    f1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154

Chosen threshold (higher F1): 0.35


**Why recall matters here:** a missed warning (false negative) costs an empty doctor slot,
while an extra call to a patient who would have come anyway (false positive) costs a minute
of front-desk time. At 0.35 the model catches far more of the real no-shows for a modest
drop in precision, so it also wins on F1.

### (d) Fit the winner on the full training set and evaluate **once** on the test set

In [12]:
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag_test = (probs >= THRESHOLD).astype(int)

print(f"Threshold: {THRESHOLD}")
print(f"Precision: {precision_score(y_test, flag_test):.3f}")
print(f"Recall   : {recall_score(y_test, flag_test):.3f}")
print(f"Flagged  : {flag_test.sum()} of {len(flag_test)}")
print(f"ROC-AUC  : {roc_auc_score(y_test, probs):.3f}")
print(f"(For information only: tuned forest test ROC-AUC = "
      f"{roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1]):.3f})")

Threshold: 0.35
Precision: 0.571
Recall   : 0.571
Flagged  : 49 of 198
ROC-AUC  : 0.789
(For information only: tuned forest test ROC-AUC = 0.741)


### (e) Save the test-set risk scores

In [13]:
scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs.round(4),
    "flag": flag_test,
})
scores.to_csv("test_risk_scores.csv", index=False)
print(scores.shape, "| rows with flag = 1:", scores["flag"].sum())
scores.sort_values("risk_prob", ascending=False).head(10)

(198, 4) | rows with flag = 1: 49


,appointment_id,no_show,risk_prob,flag
68,A0635,1,0.9372,1
24,A0544,1,0.9241,1
141,A0293,1,0.9196,1
106,A0950,1,0.8186,1
114,A0996,0,0.7667,1
69,A0067,0,0.7498,1
172,A0769,1,0.7444,1
10,A0364,0,0.7319,1
160,A0806,1,0.7035,1
173,A0560,0,0.7019,1


**How a clinic would really set the threshold:** start from how many follow-up calls the
front desk can make in an evening, and lower the threshold until the flagged list fills that
capacity, weighing the cost of a call against the cost of an empty slot.